# Dimensionality Reduction: PCA, t-SNE, and UMAP

## 1. Introduction

**Dimensionality reduction** is one of the most essential techniques in data science and machine learning. It transforms high-dimensional data into lower dimensions while preserving important information.

### What we'll learn:

- **The curse of dimensionality** - why high dimensions are problematic
- **PCA (Principal Component Analysis)** - linear dimensionality reduction through variance maximization
- **t-SNE (t-Distributed Stochastic Neighbor Embedding)** - non-linear method for visualization
- **UMAP (Uniform Manifold Approximation and Projection)** - modern, fast alternative to t-SNE

### Why dimensionality reduction matters:

1. **Visualization** - humans can't visualize beyond 3D, but data often has hundreds or thousands of dimensions
2. **Computational efficiency** - fewer dimensions = faster algorithms and less memory
3. **Noise reduction** - eliminating less important dimensions can improve model performance
4. **Feature extraction** - discovering underlying structure and patterns in data
5. **Data exploration** - understanding relationships between samples

### Key intuitions we'll build:

- How variance relates to information content
- The difference between **linear** (PCA) and **non-linear** (t-SNE, UMAP) methods
- Why **local structure** vs **global structure** preservation matters
- Trade-offs between speed, accuracy, and interpretability

## 2. Setup

Let's import the necessary libraries and set up our environment for reproducible results.

In [ ]:
# Standard library imports
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Machine learning libraries
from sklearn.datasets import load_digits, load_iris, make_swiss_roll, make_s_curve
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Try to import UMAP (it might not be installed)
try:
    from umap import UMAP
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("UMAP not available. Install with: pip install umap-learn")

# Shared library
from aiml_notebooks import set_seed

# Setup for autoreload
%load_ext autoreload
%autoreload 2

# Better plots
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

Set random seed for reproducibility across NumPy, PyTorch, and random.

In [ ]:
set_seed(42)

## 3. Understanding the Curse of Dimensionality

Before diving into solutions, let's understand the problem. The **curse of dimensionality** refers to various phenomena that arise when analyzing data in high-dimensional spaces.

### Key problems:

1. **Distance becomes meaningless** - in high dimensions, all points are roughly equidistant
2. **Exponential data requirements** - need exponentially more data to fill the space
3. **Computational cost** - operations become prohibitively expensive
4. **Overfitting** - models can easily memorize noise

Let's demonstrate with a simple experiment: how distances behave as dimensions increase.

In [ ]:
# Generate random points in different dimensions
n_samples = 1000
dimensions = [2, 10, 50, 100, 500, 1000]

results = []
for dim in dimensions:
    # Generate random points from standard normal distribution
    points = np.random.randn(n_samples, dim)
    
    # Compute pairwise distances for first 100 points (for efficiency)
    sample_points = points[:100]
    distances = []
    
    # Calculate distance from first point to all others
    for i in range(1, len(sample_points)):
        dist = np.linalg.norm(sample_points[0] - sample_points[i])
        distances.append(dist)
    
    # Calculate statistics
    mean_dist = np.mean(distances)
    std_dist = np.std(distances)
    
    results.append({
        'dim': dim,
        'mean': mean_dist,
        'std': std_dist,
        'cv': std_dist / mean_dist  # Coefficient of variation
    })

# Print results
print("Distance statistics as dimensions increase:")
print("\nDim\tMean Dist\tStd Dist\tCV (std/mean)")
print("-" * 50)
for r in results:
    print(f"{r['dim']}\t{r['mean']:.2f}\t\t{r['std']:.2f}\t\t{r['cv']:.4f}")

**Key observation**: As dimensions increase, the **coefficient of variation (CV)** decreases dramatically. This means all points become roughly the same distance apart - distances lose their discriminative power!

When CV approaches 0, it means there's little variance in distances relative to the mean. In 2D, points have meaningful near/far relationships. In 1000D, almost everything is "far" and similarly distant.

Visualize how the coefficient of variation decreases with dimensions.

In [ ]:
dims = [r['dim'] for r in results]
cvs = [r['cv'] for r in results]

plt.figure(figsize=(10, 6))
plt.plot(dims, cvs, 'o-', linewidth=2, markersize=8)
plt.xlabel('Number of Dimensions', fontsize=12)
plt.ylabel('Coefficient of Variation (std/mean)', fontsize=12)
plt.title('The Curse of Dimensionality: Distances Become Meaningless', 
          fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.tight_layout()
plt.show()

print("\n💡 The dramatic drop shows that high-dimensional data needs dimensionality reduction!")

## 4. Load Example Datasets

We'll use multiple datasets to demonstrate different aspects of dimensionality reduction:

1. **Digits dataset** (64 dimensions, 1797 samples) - real high-dimensional data
2. **Iris dataset** (4 dimensions, 150 samples) - simple example for visualization
3. **Swiss Roll** (3D manifold) - to demonstrate non-linear structure

In [ ]:
# Load digits dataset (8x8 pixel images = 64 dimensions)
digits = load_digits()
X_digits = digits.data
y_digits = digits.target

print(f"Digits dataset: {X_digits.shape[0]} samples, {X_digits.shape[1]} dimensions")
print(f"Classes: {np.unique(y_digits)}")
print(f"Shape of each image: 8x8 pixels")

Visualize a few example digits to understand our data.

In [ ]:
# Show example digits
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(f"Label: {y_digits[i]}")
    ax.axis('off')
plt.suptitle('Example Handwritten Digits (8x8 pixels = 64 dimensions)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Load the Iris dataset as a simpler example.

In [ ]:
# Load Iris dataset (4 dimensions: sepal length/width, petal length/width)
iris = load_iris()
X_iris = iris.data
y_iris = iris.target

print(f"\nIris dataset: {X_iris.shape[0]} samples, {X_iris.shape[1]} dimensions")
print(f"Features: {iris.feature_names}")
print(f"Classes: {iris.target_names}")

Create a synthetic Swiss Roll dataset to demonstrate non-linear manifolds.

In [ ]:
# Generate Swiss Roll - a 2D manifold embedded in 3D space
X_swiss, color_swiss = make_swiss_roll(n_samples=2000, noise=0.1, random_state=42)

print(f"\nSwiss Roll: {X_swiss.shape[0]} samples, {X_swiss.shape[1]} dimensions")
print("This is a 2D manifold (surface) rolled up in 3D space")

Visualize the Swiss Roll in 3D to see its structure.

In [ ]:
# Visualize Swiss Roll in 3D
fig = plt.figure(figsize=(12, 5))

ax1 = fig.add_subplot(121, projection='3d')
ax1.scatter(X_swiss[:, 0], X_swiss[:, 1], X_swiss[:, 2], 
           c=color_swiss, cmap='viridis', s=10, alpha=0.6)
ax1.set_title('Swiss Roll in 3D Space', fontsize=12, fontweight='bold')
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_zlabel('Z')

ax2 = fig.add_subplot(122)
ax2.scatter(color_swiss, np.zeros_like(color_swiss), 
           c=color_swiss, cmap='viridis', s=10, alpha=0.6)
ax2.set_title('True Underlying 1D Structure', fontsize=12, fontweight='bold')
ax2.set_xlabel('Position along roll')
ax2.set_yticks([])

plt.tight_layout()
plt.show()

print("\n💡 The Swiss Roll looks 3D, but the data actually lies on a 2D surface!")
print("Good dimensionality reduction should 'unroll' it back to 2D.")

## 5. PCA: Principal Component Analysis

**PCA** is the most widely used linear dimensionality reduction technique. It finds the directions (principal components) along which the data varies the most.

### Core idea:

1. **Variance = Information** - dimensions with high variance contain more information
2. **Find orthogonal axes** that maximize variance
3. **Project data** onto these new axes
4. **Keep top k components** that explain most variance

### Mathematical intuition:

- **Centering**: Subtract mean so data is centered at origin
- **Covariance matrix**: Compute $C = \frac{1}{n}X^TX$ where $X$ is centered data
- **Eigendecomposition**: Find eigenvectors (directions) and eigenvalues (variance) of $C$
- **Principal components**: Eigenvectors with largest eigenvalues

The $k$-th principal component is the direction that maximizes variance among all directions orthogonal to the previous $k-1$ components.

### 5.1 PCA from Scratch

Let's implement PCA from scratch to understand the mathematics.

In [ ]:
def pca_from_scratch(X, n_components=2):
    """
    PCA implementation from scratch using eigendecomposition.
    
    Args:
        X: Data matrix [n_samples, n_features]
        n_components: Number of principal components to keep
    
    Returns:
        X_reduced: Projected data [n_samples, n_components]
        components: Principal components [n_components, n_features]
        explained_var: Variance explained by each component
    """
    # Step 1: Center the data (subtract mean)
    X_centered = X - np.mean(X, axis=0)
    
    # Step 2: Compute covariance matrix
    # C = (1/n) * X^T X
    n_samples = X.shape[0]
    cov_matrix = (X_centered.T @ X_centered) / (n_samples - 1)
    
    # Step 3: Compute eigenvalues and eigenvectors
    eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)
    
    # Step 4: Sort by eigenvalues (descending)
    idx = eigenvalues.argsort()[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]
    
    # Step 5: Select top k eigenvectors (principal components)
    components = eigenvectors[:, :n_components].T  # [n_components, n_features]
    
    # Step 6: Project data onto principal components
    X_reduced = X_centered @ components.T  # [n_samples, n_components]
    
    # Step 7: Compute explained variance
    total_var = np.sum(eigenvalues)
    explained_var = eigenvalues[:n_components] / total_var
    
    return X_reduced, components, explained_var

print("✓ PCA implementation ready!")

Test our PCA implementation on the Iris dataset.

In [ ]:
# Apply our PCA implementation
X_iris_pca, components, explained_var = pca_from_scratch(X_iris, n_components=2)

print("PCA Results:")
print(f"Original shape: {X_iris.shape}")
print(f"Reduced shape: {X_iris_pca.shape}")
print(f"\nExplained variance ratio:")
print(f"  PC1: {explained_var[0]:.1%}")
print(f"  PC2: {explained_var[1]:.1%}")
print(f"  Total: {np.sum(explained_var):.1%}")
print(f"\n💡 We reduced from 4D to 2D while keeping {np.sum(explained_var):.1%} of variance!")

Visualize the PCA projection of the Iris dataset.

In [ ]:
# Plot PCA results
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_iris_pca[:, 0], X_iris_pca[:, 1], 
                     c=y_iris, cmap='viridis', s=50, alpha=0.7, edgecolors='black')
plt.xlabel(f'PC1 ({explained_var[0]:.1%} variance)', fontsize=12)
plt.ylabel(f'PC2 ({explained_var[1]:.1%} variance)', fontsize=12)
plt.title('Iris Dataset Projected onto First 2 Principal Components', 
          fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Species', ticks=[0, 1, 2])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Notice how PCA separates the species! The first two components capture")
print("   the most important patterns that distinguish between Iris species.")

### 5.2 Choosing the Number of Components

How many components should we keep? There are several strategies:

1. **Explained variance threshold** - keep components until you explain 90-95% of variance
2. **Elbow method** - look for "elbow" in scree plot
3. **Kaiser criterion** - keep components with eigenvalue > 1 (for standardized data)
4. **Task-specific** - determined by downstream task performance

Let's visualize the explained variance to see how many components we need.

In [ ]:
# Compute PCA with all components for digits dataset
X_digits_pca_full, _, explained_var_full = pca_from_scratch(X_digits, n_components=64)

# Compute cumulative explained variance
cumsum_var = np.cumsum(explained_var_full)

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot (individual variance)
ax1.plot(range(1, len(explained_var_full) + 1), explained_var_full, 'o-', linewidth=2)
ax1.set_xlabel('Principal Component', fontsize=12)
ax1.set_ylabel('Explained Variance Ratio', fontsize=12)
ax1.set_title('Scree Plot: Variance per Component', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.axhline(y=0.05, color='r', linestyle='--', label='5% threshold')
ax1.legend()

# Cumulative variance plot
ax2.plot(range(1, len(cumsum_var) + 1), cumsum_var, 'o-', linewidth=2, color='green')
ax2.axhline(y=0.90, color='r', linestyle='--', label='90% threshold')
ax2.axhline(y=0.95, color='orange', linestyle='--', label='95% threshold')
ax2.set_xlabel('Number of Components', fontsize=12)
ax2.set_ylabel('Cumulative Explained Variance', fontsize=12)
ax2.set_title('Cumulative Explained Variance', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()

# Find number of components for different thresholds
n_90 = np.argmax(cumsum_var >= 0.90) + 1
n_95 = np.argmax(cumsum_var >= 0.95) + 1

print(f"Components needed for 90% variance: {n_90}/{len(explained_var_full)}")
print(f"Components needed for 95% variance: {n_95}/{len(explained_var_full)}")
print(f"\n💡 We can reduce from 64 to ~{n_90} dimensions with minimal information loss!")

### 5.3 PCA with Scikit-learn

In practice, we use scikit-learn's optimized implementation. It's faster and more numerically stable for large datasets.

In [ ]:
# Use scikit-learn PCA
pca = PCA(n_components=2, random_state=42)
X_digits_pca = pca.fit_transform(X_digits)

print("Scikit-learn PCA:")
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {np.sum(pca.explained_variance_ratio_):.1%}")
print(f"\nShape: {X_digits.shape} -> {X_digits_pca.shape}")

Visualize the digits dataset in 2D using PCA.

In [ ]:
# Plot digits in 2D
plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_digits_pca[:, 0], X_digits_pca[:, 1], 
                     c=y_digits, cmap='tab10', s=30, alpha=0.7)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)', fontsize=12)
plt.title('Digits Dataset: PCA Projection (64D → 2D)', fontsize=14, fontweight='bold')
cbar = plt.colorbar(scatter, ticks=range(10))
cbar.set_label('Digit', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 PCA shows some clustering, but there's significant overlap.")
print("   This is because PCA is LINEAR - it can't capture non-linear patterns!")

### 5.4 PCA on Swiss Roll

Let's see how PCA handles non-linear structure. PCA should struggle with the Swiss Roll because it's a non-linear manifold.

In [ ]:
# Apply PCA to Swiss Roll
pca_swiss = PCA(n_components=2, random_state=42)
X_swiss_pca = pca_swiss.fit_transform(X_swiss)

print(f"Swiss Roll PCA:")
print(f"Explained variance: {pca_swiss.explained_variance_ratio_}")
print(f"Total: {np.sum(pca_swiss.explained_variance_ratio_):.1%}")

Compare the PCA result with the true underlying structure.

In [ ]:
# Visualize PCA on Swiss Roll
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# PCA result
scatter1 = ax1.scatter(X_swiss_pca[:, 0], X_swiss_pca[:, 1], 
                      c=color_swiss, cmap='viridis', s=10, alpha=0.6)
ax1.set_title('PCA: Linear Projection', fontsize=12, fontweight='bold')
ax1.set_xlabel('PC1')
ax1.set_ylabel('PC2')
plt.colorbar(scatter1, ax=ax1, label='True position')

# True structure (for comparison)
ax2.scatter(color_swiss, np.zeros_like(color_swiss), 
           c=color_swiss, cmap='viridis', s=10, alpha=0.6)
ax2.set_title('Ideal: Unrolled Structure', fontsize=12, fontweight='bold')
ax2.set_xlabel('Position along manifold')
ax2.set_yticks([])

plt.tight_layout()
plt.show()

print("\n💡 PCA FAILS to unroll the Swiss Roll!")
print("   The colors are jumbled - nearby points in the true structure are far apart.")
print("   This is because PCA is LINEAR and can't handle curved manifolds.")
print("\n   We need NON-LINEAR methods like t-SNE or UMAP!")

### 5.5 Key Insights: PCA

**Strengths:**
- Fast and deterministic (always same result)
- Mathematically interpretable (principal components have meaning)
- Good for linearly separable data
- Can reconstruct original data (reversible transformation)

**Limitations:**
- Only captures linear relationships
- Assumes high variance = high importance (not always true!)
- Sensitive to scaling (always standardize first)
- Can't "unroll" non-linear manifolds

**When to use PCA:**
- Data preprocessing before machine learning
- Noise reduction and feature extraction
- When you need interpretable components
- When speed matters (t-SNE/UMAP are slower)

## 6. t-SNE: t-Distributed Stochastic Neighbor Embedding

**t-SNE** is a non-linear dimensionality reduction technique designed for visualization. Unlike PCA, it can capture complex non-linear relationships.

### Core idea:

1. **Preserve local structure** - keep nearby points together
2. **Convert distances to probabilities** - in both high-D and low-D
3. **Minimize divergence** - make low-D probabilities match high-D probabilities

### How it works:

1. **High-D**: For each point, compute probability of picking neighbors based on Gaussian similarity
2. **Low-D**: Initialize random 2D positions, compute probabilities using t-distribution (heavier tails)
3. **Optimize**: Adjust 2D positions to minimize KL divergence between high-D and low-D probability distributions

### Why t-distribution?

The **t-distribution has heavier tails** than Gaussian. This solves the "crowding problem" - in 2D there's less space than high-D, so we need to push dissimilar points farther apart. The heavy tails allow this without penalizing too much.

### Key hyperparameter: perplexity

**Perplexity** roughly corresponds to the number of nearest neighbors to consider. Typical values: 5-50.
- Low perplexity (5-10): focuses on very local structure
- High perplexity (30-50): captures more global structure
- Rule of thumb: perplexity between 5 and min(50, n_samples/3)

### 6.1 t-SNE on Digits Dataset

Let's apply t-SNE to the digits dataset and compare with PCA.

In [ ]:
# Apply t-SNE
print("Running t-SNE (this may take a minute)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_digits_tsne = tsne.fit_transform(X_digits)

print(f"✓ t-SNE complete!")
print(f"Shape: {X_digits.shape} -> {X_digits_tsne.shape}")

Visualize t-SNE results and compare with PCA side-by-side.

In [ ]:
# Compare PCA vs t-SNE
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# PCA
scatter1 = ax1.scatter(X_digits_pca[:, 0], X_digits_pca[:, 1], 
                      c=y_digits, cmap='tab10', s=20, alpha=0.7)
ax1.set_title('PCA: Linear Projection', fontsize=14, fontweight='bold')
ax1.set_xlabel('PC1')
ax1.set_ylabel('PC2')
ax1.grid(True, alpha=0.3)

# t-SNE
scatter2 = ax2.scatter(X_digits_tsne[:, 0], X_digits_tsne[:, 1], 
                      c=y_digits, cmap='tab10', s=20, alpha=0.7)
ax2.set_title('t-SNE: Non-linear Embedding', fontsize=14, fontweight='bold')
ax2.set_xlabel('t-SNE 1')
ax2.set_ylabel('t-SNE 2')
ax2.grid(True, alpha=0.3)

# Shared colorbar
fig.colorbar(scatter2, ax=[ax1, ax2], ticks=range(10), label='Digit')

plt.suptitle('Digits Dataset: PCA vs t-SNE', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 HUGE DIFFERENCE!")
print("   - PCA: Significant overlap, hard to distinguish digits")
print("   - t-SNE: Clear clusters, digits are well-separated!")
print("\n   t-SNE found non-linear structure that PCA missed.")

### 6.2 Effect of Perplexity

Perplexity is the most important hyperparameter for t-SNE. Let's see how different values affect the results.

In [ ]:
# Test different perplexity values
perplexities = [5, 30, 50]
results_tsne = {}

for perp in perplexities:
    print(f"Running t-SNE with perplexity={perp}...")
    tsne = TSNE(n_components=2, random_state=42, perplexity=perp, max_iter=1000)
    results_tsne[perp] = tsne.fit_transform(X_digits)

print("\n✓ All t-SNE runs complete!")

Visualize how perplexity affects the clustering structure.

In [ ]:
# Plot different perplexities
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, perp in zip(axes, perplexities):
    scatter = ax.scatter(results_tsne[perp][:, 0], results_tsne[perp][:, 1],
                        c=y_digits, cmap='tab10', s=20, alpha=0.7)
    ax.set_title(f'Perplexity = {perp}', fontsize=12, fontweight='bold')
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.grid(True, alpha=0.3)

plt.colorbar(scatter, ax=axes, ticks=range(10), label='Digit')
plt.suptitle('Effect of Perplexity on t-SNE', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 Observations:")
print("   - Low perplexity (5): Many small, tight clusters (very local)")
print("   - Medium perplexity (30): Balanced structure (recommended)")
print("   - High perplexity (50): Broader clusters, more global structure")

### 6.3 t-SNE on Swiss Roll

Let's see if t-SNE can successfully "unroll" the Swiss Roll where PCA failed.

In [ ]:
# Apply t-SNE to Swiss Roll
print("Running t-SNE on Swiss Roll...")
tsne_swiss = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_swiss_tsne = tsne_swiss.fit_transform(X_swiss)
print("✓ Done!")

Compare PCA and t-SNE on the Swiss Roll.

In [ ]:
# Compare PCA vs t-SNE on Swiss Roll
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# PCA
scatter1 = ax1.scatter(X_swiss_pca[:, 0], X_swiss_pca[:, 1], 
                      c=color_swiss, cmap='viridis', s=10, alpha=0.6)
ax1.set_title('PCA: Failed to Unroll', fontsize=12, fontweight='bold')
ax1.set_xlabel('PC1')
ax1.set_ylabel('PC2')
plt.colorbar(scatter1, ax=ax1, label='True position')

# t-SNE
scatter2 = ax2.scatter(X_swiss_tsne[:, 0], X_swiss_tsne[:, 1], 
                      c=color_swiss, cmap='viridis', s=10, alpha=0.6)
ax2.set_title('t-SNE: Successfully Unrolled!', fontsize=12, fontweight='bold')
ax2.set_xlabel('t-SNE 1')
ax2.set_ylabel('t-SNE 2')
plt.colorbar(scatter2, ax=ax2, label='True position')

plt.suptitle('Swiss Roll: PCA vs t-SNE', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 t-SNE SUCCESS!")
print("   - Colors flow smoothly in t-SNE plot (preserves local structure)")
print("   - The manifold has been 'unrolled' from 3D to 2D")
print("   - This demonstrates t-SNE's power for non-linear structure!")

### 6.4 Key Insights: t-SNE

**Strengths:**
- Excellent for visualization - creates clear, interpretable clusters
- Captures non-linear structure (manifolds)
- Preserves local neighborhood structure very well
- Often reveals hidden patterns in complex data

**Limitations:**
- Slow for large datasets (O(n²) complexity)
- Non-deterministic (different runs give different results)
- Can't transform new data (must re-run on entire dataset)
- Distances in low-D space are not meaningful
- Global structure may be distorted
- Perplexity selection can be tricky

**When to use t-SNE:**
- Exploratory data analysis and visualization
- When you need to see clusters and groups
- With datasets of moderate size (<10,000 samples)
- When global structure is less important than local patterns

**Important caveats:**
- Cluster sizes in t-SNE plots are NOT meaningful
- Distances between clusters are NOT meaningful
- Multiple runs may give different (but equally valid) results

## 7. UMAP: Uniform Manifold Approximation and Projection

**UMAP** is a modern alternative to t-SNE that's faster and often better at preserving global structure. It's based on manifold learning and topological data analysis.

### Core idea:

1. **Build fuzzy topological representation** of high-D data
2. **Find low-D representation** with similar topology
3. **Optimize using cross-entropy** between the two representations

### Advantages over t-SNE:

- **Faster** - typically 10-100x faster than t-SNE
- **Scales better** - works on datasets with millions of points
- **Better global structure** - preserves both local AND global relationships
- **Can transform new data** - unlike t-SNE, can apply learned transformation to new points
- **More stable** - less sensitive to hyperparameters

### Key hyperparameters:

- **n_neighbors** - like perplexity in t-SNE, controls local vs global (default: 15)
- **min_dist** - minimum distance between points in low-D (default: 0.1)
  - Smaller values: tighter clusters
  - Larger values: more even distribution

### 7.1 UMAP on Digits Dataset

Let's apply UMAP and compare with PCA and t-SNE.

In [ ]:
if UMAP_AVAILABLE:
    print("Running UMAP...")
    umap = UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    X_digits_umap = umap.fit_transform(X_digits)
    print(f"✓ UMAP complete!")
    print(f"Shape: {X_digits.shape} -> {X_digits_umap.shape}")
else:
    print("⚠ UMAP not available. Install with: pip install umap-learn")
    X_digits_umap = None

Compare all three methods: PCA, t-SNE, and UMAP.

In [ ]:
if UMAP_AVAILABLE:
    # Compare all three methods
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # PCA
    scatter1 = axes[0].scatter(X_digits_pca[:, 0], X_digits_pca[:, 1], 
                              c=y_digits, cmap='tab10', s=15, alpha=0.7)
    axes[0].set_title('PCA (Linear)', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('PC1')
    axes[0].set_ylabel('PC2')
    axes[0].grid(True, alpha=0.3)
    
    # t-SNE
    scatter2 = axes[1].scatter(X_digits_tsne[:, 0], X_digits_tsne[:, 1], 
                              c=y_digits, cmap='tab10', s=15, alpha=0.7)
    axes[1].set_title('t-SNE (Non-linear, Local)', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('t-SNE 1')
    axes[1].set_ylabel('t-SNE 2')
    axes[1].grid(True, alpha=0.3)
    
    # UMAP
    scatter3 = axes[2].scatter(X_digits_umap[:, 0], X_digits_umap[:, 1], 
                              c=y_digits, cmap='tab10', s=15, alpha=0.7)
    axes[2].set_title('UMAP (Non-linear, Local+Global)', fontsize=12, fontweight='bold')
    axes[2].set_xlabel('UMAP 1')
    axes[2].set_ylabel('UMAP 2')
    axes[2].grid(True, alpha=0.3)
    
    plt.colorbar(scatter3, ax=axes, ticks=range(10), label='Digit')
    plt.suptitle('Comparison: PCA vs t-SNE vs UMAP on Digits', 
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Comparison:")
    print("   - PCA: Fast but linear, overlapping clusters")
    print("   - t-SNE: Excellent local structure, very distinct clusters")
    print("   - UMAP: Similar to t-SNE but faster, better global structure")
else:
    print("Skipping comparison - UMAP not available")

### 7.2 UMAP on Swiss Roll

Let's see how UMAP handles the non-linear manifold.

In [ ]:
if UMAP_AVAILABLE:
    print("Running UMAP on Swiss Roll...")
    umap_swiss = UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    X_swiss_umap = umap_swiss.fit_transform(X_swiss)
    print("✓ Done!")
else:
    print("UMAP not available")
    X_swiss_umap = None

Compare all three methods on the Swiss Roll.

In [ ]:
if UMAP_AVAILABLE:
    # Compare all three on Swiss Roll
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # PCA
    scatter1 = axes[0].scatter(X_swiss_pca[:, 0], X_swiss_pca[:, 1], 
                              c=color_swiss, cmap='viridis', s=10, alpha=0.6)
    axes[0].set_title('PCA: Failed', fontsize=12, fontweight='bold')
    plt.colorbar(scatter1, ax=axes[0], label='True position')
    
    # t-SNE
    scatter2 = axes[1].scatter(X_swiss_tsne[:, 0], X_swiss_tsne[:, 1], 
                              c=color_swiss, cmap='viridis', s=10, alpha=0.6)
    axes[1].set_title('t-SNE: Success', fontsize=12, fontweight='bold')
    plt.colorbar(scatter2, ax=axes[1], label='True position')
    
    # UMAP
    scatter3 = axes[2].scatter(X_swiss_umap[:, 0], X_swiss_umap[:, 1], 
                              c=color_swiss, cmap='viridis', s=10, alpha=0.6)
    axes[2].set_title('UMAP: Success', fontsize=12, fontweight='bold')
    plt.colorbar(scatter3, ax=axes[2], label='True position')
    
    plt.suptitle('Swiss Roll Unrolling: PCA vs t-SNE vs UMAP', 
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Both t-SNE and UMAP successfully unroll the manifold!")
    print("   UMAP often better preserves the global shape.")
else:
    print("Skipping Swiss Roll comparison - UMAP not available")

### 7.3 Speed Comparison

One of UMAP's biggest advantages is speed. Let's measure execution time.

In [ ]:
import time

if UMAP_AVAILABLE:
    # Time each method
    methods = {}
    
    # PCA
    start = time.time()
    pca = PCA(n_components=2, random_state=42)
    pca.fit_transform(X_digits)
    methods['PCA'] = time.time() - start
    
    # t-SNE
    start = time.time()
    tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
    tsne.fit_transform(X_digits)
    methods['t-SNE'] = time.time() - start
    
    # UMAP
    start = time.time()
    umap = UMAP(n_components=2, random_state=42, n_neighbors=15)
    umap.fit_transform(X_digits)
    methods['UMAP'] = time.time() - start
    
    # Plot comparison
    plt.figure(figsize=(10, 6))
    names = list(methods.keys())
    times = list(methods.values())
    colors = ['skyblue', 'coral', 'lightgreen']
    
    bars = plt.bar(names, times, color=colors, alpha=0.7, edgecolor='black')
    plt.ylabel('Time (seconds)', fontsize=12)
    plt.title('Speed Comparison on Digits Dataset (1797 samples)', 
              fontsize=14, fontweight='bold')
    plt.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar, time_val in zip(bars, times):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{time_val:.2f}s',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\nTiming Results:")
    for name, t in methods.items():
        print(f"  {name}: {t:.2f}s")
    
    speedup = methods['t-SNE'] / methods['UMAP']
    print(f"\n💡 UMAP is {speedup:.1f}x faster than t-SNE on this dataset!")
    print("   The difference grows even larger with bigger datasets.")
else:
    print("Skipping speed comparison - UMAP not available")

### 7.4 Key Insights: UMAP

**Strengths:**
- Much faster than t-SNE (especially on large datasets)
- Better preservation of global structure
- Can transform new data points (has .transform() method)
- Scales to very large datasets (millions of points)
- More stable across different runs
- Works well in higher dimensions (3D, 4D, etc.)

**Limitations:**
- Still slower than PCA
- Mathematical foundation is more complex
- Less widely adopted than t-SNE (but growing fast)
- Still non-deterministic (though more stable)

**When to use UMAP:**
- Large datasets where t-SNE is too slow
- When you need both local and global structure
- When you need to transform new data
- General-purpose non-linear dimensionality reduction

**UMAP vs t-SNE:**
- Use UMAP by default (faster, more scalable)
- Use t-SNE if you specifically need very tight local clustering
- Both are excellent for visualization

## 8. Practical Guidelines: Which Method to Use?

Here's a decision tree for choosing the right dimensionality reduction method:

### Use PCA when:
- You need speed and deterministic results
- Interpretability matters (principal components have meaning)
- You need to reconstruct original data
- Preprocessing for machine learning
- Data is approximately linear

### Use t-SNE when:
- Primary goal is visualization
- Dataset size is moderate (<10,000 samples)
- You want very distinct clusters
- Local structure is more important than global
- You don't need to transform new data

### Use UMAP when:
- Dataset is large (>10,000 samples)
- You need both local and global structure
- Speed matters
- You need to transform new data
- General-purpose non-linear reduction

### Common workflow:

1. **Start with PCA** - fast baseline, helps understand dimensionality
2. **Try UMAP** - if PCA doesn't show clear structure
3. **Fine-tune** - adjust hyperparameters based on results
4. **Validate** - ensure patterns make sense with domain knowledge

### 8.1 Practical Example: Full Workflow

Let's demonstrate a complete workflow for exploratory data analysis.

In [ ]:
# Step 1: Standardize data (important!)
scaler = StandardScaler()
X_digits_scaled = scaler.fit_transform(X_digits)

print("Step 1: Data standardization ✓")
print(f"  Mean: {X_digits_scaled.mean():.6f}")
print(f"  Std: {X_digits_scaled.std():.6f}")

Apply PCA first to understand how much dimensionality can be reduced.

In [ ]:
# Step 2: PCA to determine intrinsic dimensionality
pca_full = PCA(n_components=0.95)  # Keep 95% variance
X_pca_reduced = pca_full.fit_transform(X_digits_scaled)

print("\nStep 2: PCA dimensionality analysis ✓")
print(f"  Original dimensions: {X_digits.shape[1]}")
print(f"  Components for 95% variance: {X_pca_reduced.shape[1]}")
print(f"  Dimensionality reduction: {(1 - X_pca_reduced.shape[1]/X_digits.shape[1]):.1%}")

Apply UMAP for final visualization (using PCA-reduced data for speed).

In [ ]:
# Step 3: UMAP on PCA-reduced data (common speedup trick)
if UMAP_AVAILABLE:
    print("\nStep 3: UMAP visualization ✓")
    umap_final = UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    X_final = umap_final.fit_transform(X_pca_reduced)
    
    # Visualize
    plt.figure(figsize=(12, 8))
    scatter = plt.scatter(X_final[:, 0], X_final[:, 1], 
                         c=y_digits, cmap='tab10', s=30, alpha=0.7, edgecolors='black', linewidth=0.5)
    plt.xlabel('UMAP 1', fontsize=12)
    plt.ylabel('UMAP 2', fontsize=12)
    plt.title('Digits Dataset: Full Pipeline (64D → 21D → 2D)', 
              fontsize=14, fontweight='bold')
    cbar = plt.colorbar(scatter, ticks=range(10))
    cbar.set_label('Digit', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\n💡 Best practice: PCA first (fast preprocessing), then UMAP (visualization)")
    print("   This combines speed of PCA with quality of UMAP!")
else:
    print("\nStep 3: Skipped (UMAP not available)")

## 9. Key Takeaways

### 1. The Curse of Dimensionality is Real
- High-dimensional spaces are counterintuitive
- Distances become meaningless
- Dimensionality reduction is often essential

### 2. PCA: Linear and Fast
- Maximizes variance through orthogonal projections
- Best for linear relationships and preprocessing
- Interpretable components
- Can reconstruct original data

### 3. t-SNE: Non-linear Visualization
- Preserves local neighborhood structure
- Excellent for finding clusters
- Slow but creates beautiful visualizations
- Perplexity is key hyperparameter (5-50)

### 4. UMAP: Modern Alternative
- Faster than t-SNE, scales better
- Preserves both local AND global structure
- Can transform new data
- Generally recommended for non-linear reduction

### 5. Practical Workflow
1. Always standardize/normalize first
2. Start with PCA to understand dimensionality
3. Use UMAP/t-SNE for final visualization
4. Validate results with domain knowledge

### 6. Remember:
- **PCA** = fast, linear, interpretable
- **t-SNE** = slow, non-linear, local structure
- **UMAP** = fast, non-linear, local+global structure

### 7. Common Pitfalls to Avoid:
- Don't interpret cluster sizes/distances in t-SNE literally
- Always standardize before PCA
- Don't use t-SNE for datasets >50,000 samples (use UMAP)
- Remember: dimensionality reduction loses information!

## 10. Summary: When to Use Each Method

Quick reference table for choosing dimensionality reduction:

| Aspect | PCA | t-SNE | UMAP |
|--------|-----|-------|------|
| Speed | ⚡⚡⚡ Very Fast | 🐌 Slow | ⚡⚡ Fast |
| Scalability | ✓ Millions | ✗ <10K | ✓ Millions |
| Local Structure | ✗ Poor | ✓✓✓ Excellent | ✓✓ Very Good |
| Global Structure | ✓ Good | ✗ Poor | ✓✓ Very Good |
| Deterministic | ✓ Yes | ✗ No | ~ Mostly |
| Transform New Data | ✓ Yes | ✗ No | ✓ Yes |
| Interpretable | ✓ Yes | ✗ No | ✗ No |
| Reversible | ✓ Yes | ✗ No | ✗ No |
| Best For | Preprocessing | Visualization | General Purpose |

**Final Recommendation:** Start with PCA for understanding, use UMAP for visualization and analysis. Use t-SNE if you need extremely distinct local clusters and dataset is small.